# Style Diversity vs. Model Capacity

Does increasing LLM capacity reduce repetitive/groupthink behavior in multi-agent social media simulations?
This notebook walks through the results of the `style_diversity` experiment, comparing **gpt4o-mini** vs **gpt4o** across two scenarios.

In [ ]:
%matplotlib inline

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import yaml

# Matplotlib defaults
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "figure.dpi": 100,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

# --- Load study metadata ---
EXP_ROOT = Path("../experiments/style_diversity")

with open(EXP_ROOT / "study.yaml") as f:
    study = yaml.safe_load(f)

with open(EXP_ROOT / "h1_model_capacity" / "hypothesis.yaml") as f:
    hypothesis = yaml.safe_load(f)

with open(EXP_ROOT / "summary.json") as f:
    summary = json.load(f)

# --- Load all 4 eval.json files ---
CONDITIONS = ["gpt4o-mini", "gpt4o"]
SCENARIOS = ["ai_conference", "misinformation"]

evals = {}
for cond in CONDITIONS:
    for scen in SCENARIOS:
        run_dir = EXP_ROOT / "h1_model_capacity" / f"model={cond}" / scen
        # Find the single run_* directory
        run_dirs = sorted(run_dir.glob("run_*"))
        with open(run_dirs[0] / "eval.json") as f:
            evals[(cond, scen)] = json.load(f)

# Pre-aggregated cross-scenario means
metrics_by_cond = summary["metrics_by_condition"]

ALL_METRICS = list(metrics_by_cond["gpt4o-mini"].keys())
print(f"Study: {study['name']}")
print(f"Question: {study['question']}")
print(f"Hypothesis: {hypothesis['statement']}")
print(f"Loaded {len(evals)} eval files, {len(ALL_METRICS)} metrics")

ModuleNotFoundError: No module named 'matplotlib'

## Study Overview

**Hypothesis (H1):** Larger language models produce more diverse agent behavior (higher lexical diversity, lower self-BLEU, more varied actions).

- **Independent variable:** Model (`gpt4o-mini` vs `gpt4o`)
- **Prediction:** gpt4o outperforms gpt4o-mini on diversity metrics across scenarios

### Conditions

In [ ]:
# Build conditions table
print(f"{'Model':<14} {'Scenario':<18} {'Agents':>6} {'Steps':>6} {'Posts':>6} {'Replies':>8} {'Originals':>10} {'Boosts':>7}")
print("-" * 85)
for cond in CONDITIONS:
    for scen in SCENARIOS:
        s = evals[(cond, scen)]["summary"]
        print(f"{cond:<14} {scen:<18} {s['agents']:>6} {s['steps']:>6} {s['total_posts']:>6} {s['replies']:>8} {s['original_posts']:>10} {s['boosts']:>7}")

## Key Metrics Explained

We focus on four metrics that each capture a distinct dimension of linguistic diversity. All are computed per-agent from their post history (excluding boosts), then averaged.

### 1. Self-BLEU (lower = more diverse)

Each agent's posts are compared pairwise using BLEU score — the standard machine-translation metric based on n-gram overlap. For each post, we compute the average BLEU against all other posts by the same agent, using 1- through 4-gram precisions combined via geometric mean (no brevity penalty). The agent's self-BLEU is the mean across all posts.

*Intuitive form* — how much an agent copies itself:

$$\text{Self-BLEU}_a = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{N-1}\sum_{j \neq i} \text{BLEU}(p_i,\; p_j)$$

where $p_i$ are agent $a$'s posts and BLEU measures n-gram overlap (1- through 4-grams, geometric mean of precisions).

**Why it matters:** Self-BLEU directly measures *self-plagiarism*. An agent that recycles the same phrases across posts will score high. A score near 0 means the agent rarely reuses multi-word sequences.

### 2. Lexical Diversity (higher = more diverse)

The classic type-token ratio (TTR): the number of unique word types divided by the total number of word tokens across all of an agent's posts. Tokenization uses a simple `\w+` regex on lowercased text.

*Exact definition:*

$$\text{Lexical Diversity}_a = \frac{|\{\text{unique tokens}\}|}{|\text{all tokens}|}$$

over all posts by agent $a$. Ranges from $1/N$ (one word repeated $N$ times) to $1.0$ (every word unique).

**Why it matters:** TTR captures vocabulary richness. A low score means the agent relies on a small working vocabulary — a hallmark of repetitive LLM output. It complements self-BLEU by measuring diversity at the single-word level rather than the phrase level.

### 3. Opener Variety (higher = more diverse)

The fraction of an agent's posts that begin with a unique 5-word prefix. Each post's first 5 tokens are extracted as a tuple; opener variety = (number of unique openers) / (number of posts).

*Exact definition:*

$$\text{Opener Variety}_a = \frac{|\{\text{unique 5-word prefixes}\}|}{|\text{posts}|}$$

A score of 1.0 means every post starts differently; a score of $1/N$ means all $N$ posts open identically.

**Why it matters:** Formulaic openings ("That's a great point, ...") are a common LLM repetition pattern and are immediately noticeable to readers. This metric specifically targets that failure mode.

### 4. Content Evolution (higher = more diverse)

Consecutive posts (sorted by simulation step) are compared using cosine distance on term-frequency vectors. Content evolution is the mean cosine distance between each pair of adjacent posts.

*Intuitive form* — how much an agent's focus shifts from post to post:

$$\text{Content Evolution}_a = \frac{1}{N-1} \sum_{t=1}^{N-1} \left(1 - \frac{\mathbf{tf}_{t} \cdot \mathbf{tf}_{t+1}}{\|\mathbf{tf}_{t}\|\;\|\mathbf{tf}_{t+1}\|}\right)$$

where $\mathbf{tf}_t$ is the term-frequency vector of the agent's post at step $t$. The quantity inside the sum is the cosine distance: 0 if consecutive posts use identical word distributions, 1 if they share no words at all.

**Why it matters:** The previous metrics measure diversity across all posts regardless of order. Content evolution captures *temporal* diversity — whether an agent develops new arguments over time or keeps restating the same points. High evolution means the agent's focus shifts meaningfully from step to step.

## Headline Comparison

Four key metrics averaged across both scenarios. Arrows indicate which direction is "better" for diversity.

In [ ]:
headline_metrics = ["self_bleu", "lexical_diversity", "opener_variety", "content_evolution"]
# For annotation: lower is better for self_bleu, higher is better for the rest
lower_is_better = {"self_bleu"}

mini_vals = [metrics_by_cond["gpt4o-mini"][m] for m in headline_metrics]
full_vals = [metrics_by_cond["gpt4o"][m] for m in headline_metrics]

x = np.arange(len(headline_metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width / 2, mini_vals, width, label="gpt4o-mini", color="#6baed6")
bars2 = ax.bar(x + width / 2, full_vals, width, label="gpt4o", color="#e6550d")

ax.set_ylabel("Score")
ax.set_title("Headline Diversity Metrics (avg across scenarios)")
ax.set_xticks(x)
labels = []
for m in headline_metrics:
    arrow = "lower=better" if m in lower_is_better else "higher=better"
    labels.append(f"{m}\n({arrow})")
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 1.05)

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{bar.get_height():.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.show()

print("\nTakeaway: gpt4o dramatically reduces self-BLEU (0.33 -> 0.04) while nearly")
print("doubling lexical diversity and opener variety. The larger model produces")
print("substantially more varied text across all four headline metrics.")

## Full Metric Radar Chart

All 10 metrics on a radar/spider chart. Metrics are normalized so that **outward = more diverse** (repetition metrics are flipped: `1 - value`).

In [ ]:
# Metrics where lower = more repetitive, so we flip them for the radar
flip_metrics = {"self_bleu", "near_duplicate_rate", "target_fixation"}

def normalize_for_radar(values_dict):
    """Flip repetition metrics so outward = better diversity."""
    out = {}
    for m, v in values_dict.items():
        out[m] = (1 - v) if m in flip_metrics else v
    return out

mini_radar = normalize_for_radar(metrics_by_cond["gpt4o-mini"])
full_radar = normalize_for_radar(metrics_by_cond["gpt4o"])

labels_radar = ALL_METRICS
N = len(labels_radar)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

mini_r = [mini_radar[m] for m in labels_radar] + [mini_radar[labels_radar[0]]]
full_r = [full_radar[m] for m in labels_radar] + [full_radar[labels_radar[0]]]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, mini_r, "o-", linewidth=2, label="gpt4o-mini", color="#6baed6")
ax.fill(angles, mini_r, alpha=0.15, color="#6baed6")
ax.plot(angles, full_r, "o-", linewidth=2, label="gpt4o", color="#e6550d")
ax.fill(angles, full_r, alpha=0.15, color="#e6550d")

# Label each axis
display_labels = []
for m in labels_radar:
    lbl = m.replace("_", " ")
    if m in flip_metrics:
        lbl += "\n(1-x)"
    display_labels.append(lbl)
ax.set_thetagrids(np.degrees(angles[:-1]), display_labels, fontsize=8)
ax.set_ylim(0, 1)
ax.set_title("Diversity Profile (outward = better)", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

print("\nThe gpt4o profile (orange) envelops gpt4o-mini (blue) on nearly every axis.")
print("The largest gains are in self-BLEU (flipped), opener variety, and content evolution.")
print("The one exception is inter-agent distinctiveness, where gpt4o-mini scores higher --")
print("likely because its agents each fixate on different repetitive patterns.")

## Scenario Consistency

Is the model effect consistent across scenarios, or scenario-dependent?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for idx, scen in enumerate(SCENARIOS):
    ax = axes[idx]
    mini_agg = evals[("gpt4o-mini", scen)]["aggregated"]
    full_agg = evals[("gpt4o", scen)]["aggregated"]

    x = np.arange(len(ALL_METRICS))
    width = 0.35

    mini_v = [mini_agg[m] for m in ALL_METRICS]
    full_v = [full_agg[m] for m in ALL_METRICS]

    ax.barh(x - width / 2, mini_v, width, label="gpt4o-mini", color="#6baed6")
    ax.barh(x + width / 2, full_v, width, label="gpt4o", color="#e6550d")

    ax.set_yticks(x)
    ax.set_yticklabels([m.replace("_", "\n") for m in ALL_METRICS], fontsize=8)
    ax.set_xlabel("Score")
    ax.set_title(scen.replace("_", " ").title())
    ax.set_xlim(0, 1.05)
    ax.legend(fontsize=8)
    ax.invert_yaxis()

plt.suptitle("All Metrics by Scenario", fontsize=14)
plt.tight_layout()
plt.show()

print("\nThe model effect is highly consistent across both scenarios.")
print("gpt4o outperforms on diversity metrics in both ai_conference and misinformation,")
print("with similar relative magnitudes. This suggests the effect is robust to scenario content.")

## Per-Agent Distributions

Do agents under gpt4o shift the mean AND tighten variance, or just shift?

In [ ]:
strip_metrics = ["self_bleu", "lexical_diversity", "action_entropy", "opener_variety"]

# Collect per-agent values pooled across scenarios
agent_data = {cond: {m: [] for m in strip_metrics} for cond in CONDITIONS}
for cond in CONDITIONS:
    for scen in SCENARIOS:
        agents = evals[(cond, scen)]["agents"]
        for agent_name, agent_metrics in agents.items():
            for m in strip_metrics:
                agent_data[cond][m].append(agent_metrics[m])

fig, axes = plt.subplots(1, len(strip_metrics), figsize=(14, 5), sharey=True)

for idx, m in enumerate(strip_metrics):
    ax = axes[idx]
    for ci, cond in enumerate(CONDITIONS):
        vals = agent_data[cond][m]
        color = "#6baed6" if cond == "gpt4o-mini" else "#e6550d"
        # Jitter y position
        y = np.full(len(vals), ci) + np.random.default_rng(42).uniform(-0.15, 0.15, len(vals))
        ax.scatter(vals, y, alpha=0.7, color=color, edgecolors="white", s=50, label=cond if idx == 0 else None)
        # Mean marker
        mean_val = np.mean(vals)
        ax.scatter([mean_val], [ci], marker="|", color="black", s=200, linewidths=2, zorder=5)

    ax.set_yticks([0, 1])
    ax.set_yticklabels(CONDITIONS)
    ax.set_title(m.replace("_", " "), fontsize=10)
    ax.set_xlim(-0.05, 1.05)

axes[0].legend(loc="upper left", fontsize=8)
fig.suptitle("Per-Agent Score Distributions (pooled across scenarios)", fontsize=13)
plt.tight_layout()
plt.show()

# Print mean and std
print(f"\n{'Metric':<22} {'gpt4o-mini mean (std)':>24} {'gpt4o mean (std)':>24}")
print("-" * 72)
for m in strip_metrics:
    mini_vals = agent_data["gpt4o-mini"][m]
    full_vals = agent_data["gpt4o"][m]
    print(f"{m:<22} {np.mean(mini_vals):>10.3f} ({np.std(mini_vals):.3f})     {np.mean(full_vals):>10.3f} ({np.std(full_vals):.3f})")

print("\ngpt4o both shifts the mean (better diversity) AND substantially tightens")
print("the agent-level variance, especially for self_bleu and opener_variety.")
print("This means ALL agents benefit, not just a few high-performers.")

## Action Behavior Breakdown

What types of actions do agents take? gpt4o-mini is locked into reply-only mode, while gpt4o shows a richer behavioral repertoire.

In [ ]:
# Aggregate action counts across scenarios per condition
action_totals = {}
for cond in CONDITIONS:
    totals = {"replies": 0, "original_posts": 0, "boosts": 0}
    for scen in SCENARIOS:
        s = evals[(cond, scen)]["summary"]
        totals["replies"] += s["replies"]
        totals["original_posts"] += s["original_posts"]
        totals["boosts"] += s["boosts"]
    action_totals[cond] = totals

action_types = ["replies", "original_posts", "boosts"]
colors = ["#6baed6", "#fd8d3c", "#74c476"]

fig, ax = plt.subplots(figsize=(10, 3.5))

y_pos = np.arange(len(CONDITIONS))
left = np.zeros(len(CONDITIONS))

for i, action in enumerate(action_types):
    counts = [action_totals[cond][action] for cond in CONDITIONS]
    bars = ax.barh(y_pos, counts, left=left, color=colors[i], label=action.replace("_", " "))
    # Label each segment
    for j, (c, l) in enumerate(zip(counts, left)):
        if c > 0:
            ax.text(l + c / 2, j, str(c), ha="center", va="center", fontsize=10, fontweight="bold")
    left += counts

ax.set_yticks(y_pos)
ax.set_yticklabels(CONDITIONS)
ax.set_xlabel("Number of Actions (pooled across scenarios)")
ax.set_title("Action Type Breakdown")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

print("\ngpt4o-mini: 100% replies (135/135 model actions). Zero original posts, zero boosts.")
print("gpt4o: 109 replies + 19 original posts + 2 boosts = 130 model actions.")
print("\nThe smaller model is structurally locked into reply-only behavior, while the")
print("larger model spontaneously creates original content and engages in boosting.")
print("This is a qualitative behavioral difference, not just a quantitative one.")

## Takeaways

### Key Findings

- **Self-BLEU drops 10x** (0.33 to 0.04): gpt4o agents almost never repeat themselves, while gpt4o-mini agents recycle phrases heavily.
- **Lexical diversity nearly doubles** (0.29 to 0.49): gpt4o uses a much richer vocabulary.
- **Opener variety doubles** (0.48 to 0.95): gpt4o-mini opens posts with a handful of stock phrases; gpt4o is nearly unique every time.
- **Content evolution jumps 2x** (0.27 to 0.61): gpt4o agents develop their arguments over time rather than repeating the same points.
- **Action repertoire unlocked**: gpt4o produces original posts and boosts, while gpt4o-mini is locked into reply-only mode (100% replies).
- **Effect is consistent across scenarios**: both ai_conference and misinformation show the same pattern, suggesting the effect is model-driven, not scenario-driven.
- **Per-agent variance tightens**: gpt4o doesn't just shift the mean -- ALL agents improve, reducing the gap between most and least diverse agents.
- **Inter-agent distinctiveness is the exception**: gpt4o-mini scores higher here, likely because each agent fixates on *different* repetitive patterns.

### Limitations

- **n=1 run per condition**: no seed variation, so we cannot assess run-to-run stability.
- **Two scenarios only**: broader generalization requires more scenario types.
- **Cost confound**: gpt4o is ~10x more expensive per token; the diversity gains must be weighed against inference cost.
- **No temperature variation**: all runs use default temperature; higher temperature on mini might partially close the gap.

### Next Steps

- Add seed replication (n>=3) to assess statistical significance.
- Test intermediate models (gpt4o-mini with higher temperature, or a mid-tier model) to map the capacity-diversity curve.
- Run the misinformation scenario with gpt4o to see if it also produces original posts (currently 0 originals in both models for misinformation).
- Add human evaluation of post quality to complement automated diversity metrics.